# 07 — Build metadata tables for the full time series

**Purpose.** The inference twin of notebook 06. Turns the raw output of the time-series pull
(notebook 05) into four flat CSVs, one row per scene, keyed on `file_name`. In Athena they become the
`*_full_time_series_250` tables, which `sql/inference_data_query.sql` filters to the scenes good enough
for inference (cloud cover, season, pixel count, no-data and zero pixels).

1. `processed_logs_250.csv`: parsed from the pull log (pull date, `glims_id`, CRS, cloud cover,
   scene time, image quality).
2. `file_attributes_250.csv`: from each file's name and size (date, satellite, bytes).
3. `image_attributes_250.csv`: from opening each GeoTIFF (shape, bounds, band count, EPSG,
   resolution, and counts of zero, no-data and negative pixels). Unreadable files get no row.
4. `ee_metadata_250.csv`: Earth Engine's metadata, flattened; the first record is kept when Earth
   Engine returns more than one per scene.

The code is in `glacierview.metadata`. Athena reads these CSVs by column position, so the column
order there is part of the contract.

## Inputs
All under `config.landing_zone(config.INFERENCE_DATA_LABEL)`. To read an external disk instead, start
Jupyter with `GV_DATA_ROOT` pointing at it.
- **`landsat/<glims_id>/*.tif`** and **`meta_data/metadata_list_l{5,7,8}`**: from notebook 05. A fresh
  clone has none; the full set (about 486 GB) is in `s3://full-time-series-images-t1-l2-sr`
  (us-west-1). Every file is read in full, so this is slow.
- **`logs/log_geog_area_rollup_250.log`**: the pull log from notebook 05.

## Outputs
- The four CSVs in `config.processed_metadata_dir(config.INFERENCE_DATA_LABEL)`. Gitignored.

## Manual step afterwards (not in code)
Upload each CSV to its own prefix in `s3://full-time-series-metadata-t1-l2-sr` (us-east-1), one per
Glue table (`{processed_logs,file_attributes,image_attributes,ee_metadata}_full_time_series_250/`). The
Glue catalog is the `glacier-view` database in **us-west-1**. See `.agents/references/aws.md`.

In [ ]:
from glacierview import config, log
from glacierview import metadata as md

log.configure()
LABEL = config.INFERENCE_DATA_LABEL
landsat = config.landsat_dir(LABEL)
out = config.processed_metadata_dir(LABEL)
out.mkdir(parents=True, exist_ok=True)
glims_ids = md.glacier_ids(landsat)

In [ ]:
logs = md.parse_pull_log(config.log_dir(LABEL) / "log_geog_area_rollup_250.log")
logs[logs["glims_id"].isin(glims_ids)].to_csv(out / "processed_logs_250.csv", index=False)

In [ ]:
files, images = md.scan_scenes(landsat, glims_ids)
files.to_csv(out / "file_attributes_250.csv", index=False)
images.to_csv(out / "image_attributes_250.csv", index=False)

In [ ]:
ee_metadata = md.read_ee_metadata(landsat, glims_ids)
ee_metadata.to_csv(out / "ee_metadata_250.csv", index=False)